# TERAKOYAの自己紹介を自動入力しよう

Seleniumを使ってSAMURAI TERAKOYAへログインし、自己紹介を自動更新します。

In [1]:
# Seleniumとwebdriver-managerをインストール
!pip install -q selenium
!pip install -q webdriver-manager

# Google Chromeをインストール
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get update -qq
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 8.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatk1.0-data.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3build1_amd64.deb ...
Unpacking libatk1.0-0:amd64 (2.36.0-3build1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../02-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd6

In [2]:
# 必要なライブラリをインポート
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from getpass import getpass

# Chromeをヘッドレスモードで起動
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

service = Service(ChromeDriverManager().install())
chrome_driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(chrome_driver, 30)

In [5]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from getpass import getpass
import time

# TERAKOYAを開く
chrome_driver.get("https://terakoya.sejuku.net/register")

# ログイン画面を表示
login_link = wait.until(
    EC.element_to_be_clickable(
        (By.XPATH, "//*[contains(text(), 'ログイン')]")
    )
)
login_link.click()

# メールアドレスとパスワードを入力
email_address = input("メールアドレスを入力してください: ")
password = getpass("パスワードを入力してください: ")

# ログイン用の入力欄を取得
email_inputs = chrome_driver.find_elements(By.NAME, "email")
password_inputs = chrome_driver.find_elements(By.NAME, "password")

login_email = email_inputs[-1]
login_password = password_inputs[-1]

login_email.clear()
login_email.send_keys(email_address)

login_password.clear()
login_password.send_keys(password)

# ログインボタンをクリック
login_button = wait.until(
    lambda driver: next(
        (
            b for b in driver.find_elements(By.TAG_NAME, "button")
            if b.text.strip() == "ログイン"
            and b.is_enabled()
        ),
        False
    )
)

login_button.click()

# ログイン処理が完了するまで待つ
time.sleep(5)

print("ログイン完了")

メールアドレスを入力してください:  taiki.higashino@dpp.co.jp
パスワードを入力してください: ··········
ログイン完了


In [6]:
# アカウント設定ページを開く
chrome_driver.get("https://terakoya.sejuku.net/account/profile")

# 「編集」ボタンが表示されるまで待つ
edit_button = wait.until(
    EC.element_to_be_clickable(
        (By.XPATH, "//button[normalize-space()='編集']")
    )
)

edit_button.click()

# 自己紹介欄を取得
self_intro = wait.until(
    EC.visibility_of_element_located(
        (By.XPATH, "//*[normalize-space()='自己紹介']/following::textarea[1]")
    )
)

# 自己紹介を入力
self_intro.clear()
self_intro.send_keys(
    "プログラミング学習中です！今はスクレイピングに挑戦しています！"
)

# 更新する
update_button = wait.until(
    EC.element_to_be_clickable(
        (By.XPATH, "//button[normalize-space()='更新する']")
    )
)

update_button.click()

print("自己紹介を更新しました")

自己紹介を更新しました


In [7]:
# ブラウザを終了
chrome_driver.quit()